# FunctionGemma 270M × TinyCeNN — Integrated Memory V1

Tests TinyCeNN bounded memory on `vtava/functiongemma-270m-it-simple-tool-calling`.

FunctionGemma/Gemma3 has 18 decoder layers. Full/global attention is at layers **5, 11, 17**; the other 15 layers remain the model's native 512-token sliding attention.

- conservative CeNN: replace full-attention layers **5 + 17**
- expanded CeNN: replace **5 + 11 + 17**
- matched Transformer-readout controls use the same layer sets
- selection uses validation NLL only
- test data is not used for model selection


In [ ]:
import os, sys, json, subprocess, tempfile, shutil
from pathlib import Path
from datetime import datetime, timezone

REPO = Path(tempfile.mkdtemp(prefix="functiongemma-cenn-")) / "TinyCeNN-LM"
subprocess.run(["git","clone","--quiet","https://github.com/vtavakkoli/TinyCeNN-LM.git",str(REPO)], check=True)
subprocess.run(["git","checkout","main"], cwd=REPO, check=True)

subprocess.run([
    sys.executable,"-m","pip","install","-q",
    "transformers==4.57.6","huggingface_hub==0.36.2","datasets>=3,<5",
    "pytest","pandas","matplotlib"
], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO),"--no-deps"], check=True)

os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO), str(REPO/"src")])
sys.path[:0] = [str(REPO), str(REPO/"src")]

import torch
from transformers import AutoConfig

MODEL_ID = "vtava/functiongemma-270m-it-simple-tool-calling"
cfg = AutoConfig.from_pretrained(MODEL_ID).get_text_config(decoder=True)
FULL = [i for i,t in enumerate(cfg.layer_types) if t=="full_attention"]

print("Source:", subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Architecture:", cfg.model_type)
print("Layers:", cfg.num_hidden_layers)
print("Full attention:", FULL)
print("Sliding window:", cfg.sliding_window)

assert cfg.model_type == "gemma3_text"
assert FULL == [5,11,17]


## Configuration


In [ ]:
PROFILE = "balanced" # @param ["smoke","balanced","extended"]
SAVE_TO_DRIVE = True # @param {type:"boolean"}

P = {
    "smoke": dict(
        train_contexts="64,128", test_contexts="64,128,256",
        block_size=16, features=32,
        train_documents=4, validation_documents=2, test_documents=2, warm_documents=2,
        warm_steps=2, joint_steps=4, eval_every=2,
        timing_documents=1, timing_repeats=1, decode_tokens=8, loss_chunk=8
    ),
    "balanced": dict(
        train_contexts="128,256,512", test_contexts="128,256,512,1024,2048",
        block_size=32, features=64,
        train_documents=64, validation_documents=8, test_documents=16, warm_documents=8,
        warm_steps=50, joint_steps=150, eval_every=25,
        timing_documents=2, timing_repeats=2, decode_tokens=32, loss_chunk=16
    ),
    "extended": dict(
        train_contexts="256,512,1024", test_contexts="256,512,1024,2048,4096",
        block_size=32, features=96,
        train_documents=128, validation_documents=16, test_documents=32, warm_documents=16,
        warm_steps=100, joint_steps=300, eval_every=50,
        timing_documents=3, timing_repeats=3, decode_tokens=48, loss_chunk=16
    ),
}

if not torch.cuda.is_available() and PROFILE != "smoke":
    raise RuntimeError("Select a GPU runtime or use smoke.")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/TinyCeNN/functiongemma-integrated-v1")
else:
    BASE = Path("/content/functiongemma-integrated-v1")

BASE.mkdir(parents=True, exist_ok=True)
run_id = PROFILE + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT = BASE / run_id
LOG = BASE / (run_id + ".log")
RUN = dict(P[PROFILE], seed=2029)

print(json.dumps(RUN, indent=2))
print("Results:", OUT)


## Preflight

This cell first synchronizes the temporary clone to the latest `main`. This is important if the notebook was opened before a compatibility fix was pushed.


In [ ]:
# Always test the newest repository code, even in an already-open Colab runtime.
subprocess.run(["git","fetch","origin","main"], cwd=REPO, check=True)
subprocess.run(["git","reset","--hard","origin/main"], cwd=REPO, check=True)

source = subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO, text=True).strip()
print("Testing source:", source)

env = dict(os.environ, CUDA_VISIBLE_DEVICES="", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1")
r = subprocess.run(
    [sys.executable,"-m","pytest","-q","tests/test_gemma3_integrated_memory.py"],
    cwd=REPO, env=env, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(r.stdout)
if r.returncode:
    raise RuntimeError(f"Gemma3 preflight failed: {r.returncode}")

print("✅ Gemma3 preflight passed")


## Train + evaluate


In [ ]:
cmd = [
    sys.executable, "-u",
    str(REPO/"scripts/benchmark_functiongemma_integrated_memory.py"),
    "--base-model", MODEL_ID,
    "--output-dir", str(OUT),
]
for k,v in RUN.items():
    cmd += ["--" + k.replace("_","-"), str(v)]

print(" ".join(cmd))

try:
    with LOG.open("w") as log:
        with subprocess.Popen(
            cmd, cwd=REPO, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1
        ) as p:
            for line in p.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            status = p.wait()
    if status:
        raise RuntimeError(f"Run failed: {status}")
finally:
    if OUT.exists() and LOG.exists():
        shutil.copy2(LOG, OUT/"console.log")
        archive = shutil.make_archive(str(OUT)+"-results","zip",root_dir=OUT)
        print("Archive:", archive)


## Results


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

s = pd.read_csv(OUT/"integrated_summary.csv")
selected = json.loads((OUT/"selection.json").read_text())["selected"]

cols = [c for c in [
    "candidate","context","test_perplexity","ppl_ratio","adapted_ppl_ratio",
    "total_cache_ratio","prefill_speedup","decode_speedup",
    "remaining_full_attention_layers","selected_on_validation"
] if c in s.columns]

print("Locked validation selection:", selected)
display(s[cols].sort_values(["candidate","context"]).reset_index(drop=True))

x = s[s.candidate == selected].sort_values("context")
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(x.context, x.ppl_ratio, marker="o", label="PPL ratio")
ax.plot(x.context, x.total_cache_ratio, marker="s", label="cache ratio")
ax.axhline(1, linestyle="--")
ax.set_xscale("log", base=2)
ax.set_xlabel("Context")
ax.legend()
ax.set_title(selected)
plt.show()


## Tool-calling preservation — original vs selected CeNN


In [ ]:
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
from tinycenn_lm.gemma3_integrated_memory import (
    restore_student, inference_mode, greedy_generate, native_dtype
)

M = json.loads((OUT/"manifest.json").read_text())
R = json.loads((OUT/"integrated_report.json").read_text())
S = json.loads((OUT/"selection.json").read_text())["selected"]
rec = next(r for r in R["candidates"] if r["candidate"] == S)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dt = native_dtype(dev)

tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=M["model_revision"])
teacher = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=M["model_revision"], dtype=dt, attn_implementation="sdpa"
).to(dev).eval()

student = restore_student(
    teacher,
    torch.load(OUT/rec["checkpoint"], map_location="cpu", weights_only=True)
).to(dev).eval()

@torch.no_grad()
def generate_original(ids, n=48):
    cache = DynamicCache(config=teacher.config)
    pieces = []
    token = teacher(input_ids=ids, past_key_values=cache, use_cache=True).logits[:,-1].argmax(-1, keepdim=True)
    for _ in range(n):
        pieces.append(token)
        token = teacher(input_ids=token, past_key_values=cache, use_cache=True).logits[:,-1].argmax(-1, keepdim=True)
    return torch.cat(pieces, 1)

tools = [
    {
        "type":"function",
        "function":{
            "name":"get_current_temperature",
            "description":"Get temperature for a city",
            "parameters":{
                "type":"object",
                "properties":{"location":{"type":"string"}},
                "required":["location"]
            }
        }
    },
    {
        "type":"function",
        "function":{
            "name":"calculate",
            "description":"Calculate an expression",
            "parameters":{
                "type":"object",
                "properties":{"expression":{"type":"string"}},
                "required":["expression"]
            }
        }
    }
]

tests = [
    "What's the temperature in Vienna?",
    "Calculate 18 times 7.",
]

for prompt in tests:
    messages = [
        {"role":"developer","content":"You are a model that can do function calling with the following functions"},
        {"role":"user","content":prompt},
    ]
    ids = tok.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True,
        tokenize=True, return_dict=True, return_tensors="pt"
    )["input_ids"].to(dev)

    original_ids = generate_original(ids)

    with inference_mode(student, str(dt).removeprefix("torch.")):
        cenn_ids, _ = greedy_generate(student, ids, 48)

    A = tok.decode(original_ids[0], skip_special_tokens=False)
    B = tok.decode(cenn_ids[0], skip_special_tokens=False)

    pattern = r"call:([A-Za-z_][A-Za-z0-9_]*)"
    def function_name(text):
        m = re.search(pattern, text)
        return m.group(1) if m else None

    print("\n" + "="*90)
    print("PROMPT:", prompt)
    print("\nORIGINAL:\n", A)
    print("\nCeNN:\n", B)
    print("\nFunction:", function_name(A), "vs", function_name(B))
